# Data Collection

This notebook downloads:
1. Cryptocurrency market data (BTC, ETH, BNB, ADA, SOL, DOGE) from Binance
2. Fear & Greed sentiment index
3. FRED macroeconomic indicators

Data period: 2017-01-01 to 2025-01-01
Output: CSV files in data/ folder

## Cryptocurrency Market Data
OHLCV(BTC, ETH, BNB, ADA, SOL, DOGE)

In [ ]:
import ccxt
import pandas as pd
import requests
import time

SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'ADA/USDT', 'SOL/USDT', 'DOGE/USDT']
START_DATE = '2018-02-01'
END_DATE = '2026-01-01'
FRED_API_KEY = '733699f7ea3be2264ec38699f6f8e6d5'

exchange = ccxt.binance({'enableRateLimit': True})

def download_crypto(symbol):
    since = exchange.parse8601(f'{START_DATE}T00:00:00Z')
    end = exchange.parse8601(f'{END_DATE}T23:59:59Z')
    data = []
    
    print(f"Downloading {symbol}...")
    
    while since < end:
        ohlcv = exchange.fetch_ohlcv(symbol, '1d', since, limit=1000)
        if not ohlcv:
            break
        data.extend(ohlcv)
        since = ohlcv[-1][0] + 1
        if len(ohlcv) < 1000:
            break
        time.sleep(exchange.rateLimit / 1000)
    
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df[(df['timestamp'] >= START_DATE) & (df['timestamp'] <= END_DATE)]
    df['symbol'] = symbol.replace('/USDT', '')
    
    return df

all_data = [download_crypto(symbol) for symbol in SYMBOLS]
crypto_df = pd.concat(all_data, ignore_index=True).sort_values(['symbol', 'timestamp'])
crypto_df.to_csv('data/crypto_market_data.csv', index=False)
print(f"Saved crypto_market_data.csv: {len(crypto_df):,} rows\n")

Saved crypto_market_data.csv: 15,836 rows



## Sentiment Data
CoinMarketCap Crypto Fear and Greed Index
- Metrics for crypto market trader sentiments (e.g. extreme fear, neutral, greed)

In [9]:
response = requests.get("https://api.alternative.me/fng/?limit=0&format=json")
fg = pd.DataFrame(response.json()['data'])
fg['date'] = pd.to_datetime(fg['timestamp'].astype(int), unit='s')
fg['value'] = pd.to_numeric(fg['value'])
fg = fg[(fg['date'] >= START_DATE) & (fg['date'] <= END_DATE)]
fg = fg[['date', 'value', 'value_classification']].sort_values('date').reset_index(drop=True)
fg.to_csv('data/sentiment_data.csv', index=False)
print(f"Saved sentiment_data.csv: {len(fg):,} rows\n")

Saved sentiment_data.csv: 2,888 rows



## Macroeconomic Data
FRED (Federal Reserve Economic Data)
- Federal Funds Effective Rate, 10 Year Treasury Yield

In [12]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key='733699f7ea3be2264ec38699f6f8e6d5')

macro = pd.DataFrame({
    'fed_funds': fred.get_series('DFF', observation_start='2017-01-01'),
    'treasury_10y': fred.get_series('DGS10', observation_start='2017-01-01'),
    'vix': fred.get_series('VIXCLS', observation_start='2017-01-01')
})

macro.index.name = 'date'
macro.to_csv('data/macro_data.csv')
print(f"Saved macro_data.csv: {len(macro)} rows")

Saved macro_data.csv: 3330 rows
